# 03 — Top-Down Trend Scanning\n\nScan the KB-Trends pool for technology megatrends relevant to regulatory frequency monitoring.\nFor each trend, assess its specific impact on the domain.\nOutput: Trend-derived Technology Drivers.

In [1]:
import sys, os, json
sys.path.insert(0, os.path.abspath(".."))

from src.config import CHROMA_PERSIST_DIR, MAX_RAG_CHUNKS
from src.llm import embed, safe_chat_json
from src.traceability import TechDriver, DriverOrigin, DriverConfidence, KBPool
from src import prompts

import chromadb

client = chromadb.PersistentClient(path=CHROMA_PERSIST_DIR)
collection = client.get_collection("knowledge_base")

print(f"ChromaDB: {collection.count()} documents")

ChromaDB: 30 documents


In [2]:
def retrieve_trends(query: str, n: int = MAX_RAG_CHUNKS) -> list[dict]:
    query_emb = embed([query])[0]
    results = collection.query(
        query_embeddings=[query_emb],
        n_results=n,
        where={"pool": "trend"},
        include=["documents", "metadatas"],
    )
    out = []
    for i in range(len(results["ids"][0])):
        out.append({
            "chunk_id": results["ids"][0][i],
            "content": results["documents"][0][i],
            "source_title": results["metadatas"][0][i]["source_title"],
        })
    return out


def format_rag_chunks(chunks: list[dict]) -> str:
    parts = []
    for c in chunks:
        parts.append(f"[Chunk ID: {c['chunk_id']}] (Source: {c['source_title']})\n{c['content']}")
    return "\n\n---\n\n".join(parts)

## Step 1: Scan for Technology Trends

In [3]:
# scan across multiple queries to cover different trend angles
scan_queries = [
    "AI machine learning spectrum monitoring future trends",
    "quantum sensing RF detection new technology",
    "6G spectrum management cognitive radio",
    "space satellite spectrum monitoring",
    "edge computing distributed sensor networks",
]

all_trends = []
all_source_chunk_ids = []

for query in scan_queries:
    rag_chunks = retrieve_trends(query, n=4)
    rag_text = format_rag_chunks(rag_chunks)

    prompt = prompts.TREND_SCAN.format(rag_chunks=rag_text)
    result = safe_chat_json(prompt, system="You are a technology foresight analyst identifying trends relevant to regulatory spectrum monitoring.")

    chunk_ids = result.get("source_chunk_ids_used", [])
    for trend in result.get("trends", []):
        trend["source_chunk_ids"] = chunk_ids
        all_trends.append(trend)

print(f"Found {len(all_trends)} raw trends\n")
for t in all_trends:
    print(f"  [{t.get('timeframe', '?')}] {t['name']}")
    print(f"    {t['relevance'][:100]}...")
    print()

Found 33 raw trends

  [near-term to mid-term (0-10 years)] AI-Driven Dynamic Spectrum Sensing and Cognitive Radio Networks
    Regulatory frequency monitoring will benefit from enhanced real-time detection and classification of...

  [mid-term to long-term (5-15 years)] 6G and AI-Native Networks with Cognitive Radio Evolution
    Regulatory monitoring will need to adapt to more complex, AI-driven network environments with dynami...

  [mid-term (5-10 years)] Deep Learning-Based Spectrum Prediction for Dynamic Spectrum Access
    Regulators can leverage spectrum prediction to anticipate spectrum usage patterns, optimize monitori...

  [near-term to mid-term (0-10 years)] Open Radio Access Networks (ORAN) and AI/ML-Driven Network Optimization
    Regulatory monitoring can utilize ORAN’s open architecture and AI capabilities to implement more fle...

  [mid-term (5-10 years)] Federated Learning and Energy-Efficient AI Architectures for Decentralized Spectrum Monitoring
    Regulatory spe

## Step 2: Assess Impact on Regulatory Frequency Monitoring

In [4]:
import re

def normalize_name(name):
    name = name.lower().strip()
    name = re.sub(r'\s*\([^)]*\)\s*', ' ', name)
    return re.sub(r'\s+', ' ', name).strip()

# deduplicate trends by normalized name
seen_names = set()
unique_trends = []
for t in all_trends:
    key = normalize_name(t["name"])
    if key not in seen_names:
        seen_names.add(key)
        unique_trends.append(t)

print(f"After dedup: {len(unique_trends)} unique trends (from {len(all_trends)} raw)\n")

# assess impact for each
trend_drivers: list[TechDriver] = []

for trend in unique_trends:
    rag_chunks = retrieve_trends(f"{trend['name']} {trend['description']}", n=3)
    rag_text = format_rag_chunks(rag_chunks)

    prompt = prompts.TREND_IMPACT.format(
        trend_name=trend["name"],
        trend_description=trend["description"],
        rag_chunks=rag_text,
    )
    result = safe_chat_json(prompt, system="You are assessing how technology trends impact regulatory frequency monitoring.")

    impact = result.get("impact_level", "none")
    print(f"  [{impact.upper():6s}] {trend['name']}")
    print(f"    → {result.get('impact_description', '')[:100]}")

    if impact in ("high", "medium"):
        driver = TechDriver(
            name=trend["name"],
            description=f"{trend['description']}. Impact: {result.get('impact_description', '')}",
            origin=DriverOrigin.TREND,
            confidence=DriverConfidence.LOW,
            source_chunk_ids=trend.get("source_chunk_ids", []),
        )
        trend_drivers.append(driver)

print(f"\n=== {len(trend_drivers)} Trend Drivers (high/medium impact) ===")

After dedup: 28 unique trends (from 33 raw)



  [HIGH  ] AI-Driven Dynamic Spectrum Sensing and Cognitive Radio Networks
    → The integration of AI-driven dynamic spectrum sensing and cognitive radio networks significantly enh


  [HIGH  ] 6G and AI-Native Networks with Cognitive Radio Evolution
    → The evolution of 6G and AI-native networks with cognitive radio significantly transforms regulatory 


  [HIGH  ] Deep Learning-Based Spectrum Prediction for Dynamic Spectrum Access
    → Deep learning-based spectrum prediction, exemplified by advanced models like ViTransLSTM that combin


  [HIGH  ] Open Radio Access Networks (ORAN) and AI/ML-Driven Network Optimization
    → The trend of Open Radio Access Networks (ORAN) combined with AI/ML-driven network optimization signi


...

  [HIGH  ] Quantum RF Sensing
    → Quantum RF sensing using Rydberg atom-based electric field sensors significantly enhances regulatory


  [HIGH  ] Software-Defined Radio (SDR) Evolution with Machine Learning Integration
    → The evolution of Software-Defined Radio (SDR) architectures toward fully software-defined receivers 


  [HIGH  ] Digital Twin Networks for Spectrum Monitoring
    → Digital Twin Networks (DTNs) for spectrum monitoring significantly enhance regulatory frequency moni


  [HIGH  ] THz Communications and Multi-Band Spectrum Utilization
    → The trend towards THz communications and multi-band spectrum utilization significantly increases the


  [HIGH  ] Security and Anomaly Detection Using Machine Learning
    → The application of machine learning (ML) and statistical techniques for security and anomaly detecti

=== 28 Trend Drivers (high/medium impact) ===


In [5]:
# save state
trend_state = {
    "trend_drivers": [d.model_dump(mode="json") for d in trend_drivers],
    "all_trends_raw": unique_trends,
}
with open("../data/outputs/trend_state.json", "w") as f:
    json.dump(trend_state, f, indent=2)

print(f"Saved {len(trend_drivers)} trend drivers\n")
for d in trend_drivers:
    print(f"  • {d.name}")
    print(f"    {d.description[:120]}...")
    print()

Saved 28 trend drivers

  • AI-Driven Dynamic Spectrum Sensing and Cognitive Radio Networks
    The integration of AI and machine learning models such as CNNs, LSTMs, GNNs, and multi-agent reinforcement learning enab...

  • 6G and AI-Native Networks with Cognitive Radio Evolution
    The upcoming 6G standardization aims to create AI-native, self-learning intelligent networks that integrate digital and ...

  • Deep Learning-Based Spectrum Prediction for Dynamic Spectrum Access
    Deep learning models, including novel architectures combining visual self-attention and LSTM (e.g., ViTransLSTM), enable...

  • Open Radio Access Networks (ORAN) and AI/ML-Driven Network Optimization
    ORAN introduces modular, open, vendor-neutral architectures that enable deployment of AI/ML applications (xApps, rApps) ...

  • Federated Learning and Energy-Efficient AI Architectures for Decentralized Spectrum Monitoring
    Federated learning enables decentralized AI model training across distributed co